# Transcript-to-Cell Assignment and Expression Matrix

This notebook assigns Xenium transcripts to Cellpose-SAM segmented cells
and builds gene expression matrices for downstream analysis.


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cpsam_xenium_analysis import config, data_loader as dl
from cpsam_xenium_analysis.integration import CoordinateAligner, TranscriptMapper

%matplotlib inline


## 1. Load Data


In [ ]:
crop = config.CROP_ROI

# Load transcripts
transcripts = dl.load_transcripts(min_qv=20)
print(f'Loaded {len(transcripts):,} transcripts')

# Load expression matrix for gene names
xenium_matrix, xenium_cell_ids, gene_names = dl.load_expression_matrix()
print(f'Gene panel: {len(gene_names)} genes')

# Load Xenium cells for alignment
cells_df = dl.load_cells_df()
print(f'Xenium cells: {len(cells_df):,}')

# Load cellpose-sam masks
mask_morph = np.load(config.OUTPUT_DIR / 'masks_cpsam_morphology.npy')
mask_he = np.load(config.OUTPUT_DIR / 'masks_cpsam_he.npy')
print(f'CPSAM Morphology mask: {mask_morph.shape}, cells={len(np.unique(mask_morph))-1}')
print(f'CPSAM H&E mask: {mask_he.shape}, cells={len(np.unique(mask_he))-1}')


## 2. Coordinate Alignment


In [ ]:
aligner = CoordinateAligner()

# Convert transcript coordinates to pixels and crop to ROI
transcripts_px = aligner.transcripts_to_pixel_coords(transcripts, crop_roi=crop)
print(f'Transcripts in ROI: {len(transcripts_px):,}')
transcripts_px.head()


## 3. Assign Transcripts to Cells (Morphology)


In [ ]:
mapper = TranscriptMapper(gene_names)

# Morphology-based assignment
assigned_morph = mapper.assign_transcripts(transcripts_px, mask_morph)
cell_labels_morph, _, expr_morph = mapper.build_expression_matrix(assigned_morph)

print(f'Cells with expression: {len(cell_labels_morph):,}')
print(f'Expression matrix shape: {expr_morph.shape}')
print(f'Non-zero entries: {expr_morph.nnz:,}')


## 4. Compare with Xenium Original Expression


In [ ]:
# Map Xenium cells to CPSAM cells
cell_to_mask = aligner.build_cell_id_to_mask_map(cells_df, mask_morph, crop_roi=crop)

# Aggregate Xenium expression per CPSAM cell
_, _, xenium_agg = mapper.build_expression_from_xenium_matrix(
    xenium_cell_ids, xenium_matrix, cell_to_mask
)
print(f'Aggregated Xenium matrix: {xenium_agg.shape}')


In [ ]:
# Compare transcript counts per cell
cpsam_counts = np.array(expr_morph.sum(axis=1)).flatten()
xenium_counts = np.array(xenium_agg.sum(axis=1)).flatten()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(cpsam_counts, bins=50, alpha=0.6, label='CPSAM assignment')
axes[0].hist(xenium_counts, bins=50, alpha=0.6, label='Xenium aggregated')
axes[0].set_xlabel('Transcripts per cell')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].set_title('Transcript Count Distribution')

axes[1].scatter(xenium_counts, cpsam_counts, s=1, alpha=0.3)
axes[1].set_xlabel('Xenium aggregated transcripts/cell')
axes[1].set_ylabel('CPSAM assigned transcripts/cell')
axes[1].set_title('Correlation')

plt.tight_layout()
plt.show()


## 5. Visually Compare Gene Expression Spatial Patterns


In [ ]:
# Plot spatial expression of INS (Beta cell marker)
fig = dl.visualization.plots.plot_gene_expression_spatial(
    assigned_morph, 'INS', cell_mask=mask_morph,
    title='INS expression on CPSAM morphology segmentation'
)


## 6. Also Process H&E Segmentation


In [ ]:
assigned_he = mapper.assign_transcripts(transcripts_px, mask_he)
cell_labels_he, _, expr_he = mapper.build_expression_matrix(assigned_he)

print(f'CPSAM H&E cells with expression: {len(cell_labels_he):,}')
print(f'Expression matrix shape: {expr_he.shape}')


In [ ]:
# Save expression matrices
from cpsam_xenium_analysis.integration.transcript_mapping import save_expression_npz

save_expression_npz(expr_morph, cell_labels_morph, gene_names, 
    config.OUTPUT_DIR / 'expr_cpsam_morphology')
save_expression_npz(expr_he, cell_labels_he, gene_names, 
    config.OUTPUT_DIR / 'expr_cpsam_he')
print('Expression matrices saved!')
